# Day 048 — Exercise 3: fit_scaler + scale_features

**What you'll build:**
- `fit_scaler(X_train) -> StandardScaler` — fit a scaler on **training** data only
- `scale_features(scaler, X) -> pd.DataFrame` — apply the fitted scaler, return a DataFrame preserving column names and index

**Why it matters:** Features on different scales bias distance-based models and slow down gradient descent. After scaling, `area` (500–3000) and `bedrooms` (1–5) both have mean 0 and std 1, so neither dominates. Critical rule: **fit on training data only**, then apply the same transform to test data. Fitting on test data leaks information.

## Provided: Setup + prior functions

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Synthetic housing dataset with one categorical column (neighborhood)."""
    rng = np.random.default_rng(seed)
    area         = rng.uniform(500, 3000, n).round(0)
    bedrooms     = rng.integers(1, 6, n)
    age          = rng.uniform(0, 50, n).round(1)
    neighborhood = rng.choice(['downtown', 'suburb', 'rural'], n)
    price = (
        area * 150
        + bedrooms * 10_000
        - age * 1_000
        + np.where(neighborhood == 'downtown', 50_000, 0)
        + np.where(neighborhood == 'suburb',   20_000, 0)
        + rng.standard_normal(n) * 10_000
    ).round(-2)
    return pd.DataFrame({
        'area':         area.astype(int),
        'bedrooms':     bedrooms,
        'age':          age,
        'neighborhood': neighborhood,
        'price':        price.astype(int),
    })


def prepare_features(df: pd.DataFrame, target_col: str,
                     numeric_only: bool = True):
    """Return (X, y) separating features from target."""
    X = df.drop(columns=[target_col])
    if numeric_only:
        X = X.select_dtypes(include='number')
    y = df[target_col]
    return X, y


def split_data(X: pd.DataFrame, y: pd.Series,
               test_size: float = 0.2,
               random_state: int = 42) -> dict:
    """Wrap train_test_split, return a result dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return {
        'X_train':    X_train,
        'X_test':     X_test,
        'y_train':    y_train,
        'y_test':     y_test,
        'n_train':    len(X_train),
        'n_test':     len(X_test),
        'n_features': X_train.shape[1],
    }


def encode_categoricals(df: pd.DataFrame,
                         cat_cols: list | None = None,
                         drop_first: bool = False) -> pd.DataFrame:
    """One-hot encode categorical columns with pd.get_dummies."""
    if cat_cols is None:
        cat_cols = df.select_dtypes(include='object').columns.tolist()
    if not cat_cols:
        return df.copy()
    encoded = pd.get_dummies(df, columns=cat_cols, drop_first=drop_first)
    # pandas 2.x returns bool dtype for dummy columns; convert to int
    bool_cols = encoded.select_dtypes(include='bool').columns.tolist()
    for c in bool_cols:
        encoded[c] = encoded[c].astype(int)
    return encoded

## Your Implementation

In [ ]:
from sklearn.preprocessing import StandardScaler


def fit_scaler(X_train: pd.DataFrame) -> StandardScaler:
    """
    Fit a StandardScaler on training data only.
    Returns the fitted scaler (use it later to transform test data).
    """
    # TODO: scaler = StandardScaler()
    # TODO: scaler.fit(X_train)
    # TODO: return scaler
    pass


def scale_features(scaler: StandardScaler,
                   X: pd.DataFrame) -> pd.DataFrame:
    """
    Transform X using a fitted StandardScaler.
    Returns a DataFrame with the same column names and index as X.
    """
    # TODO: scaled = scaler.transform(X)
    # TODO: return pd.DataFrame(scaled, columns=X.columns, index=X.index)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df      = make_regression_data(100)
    encoded = encode_categoricals(df)
    X, y    = prepare_features(encoded, 'price', numeric_only=False)
    split   = split_data(X, y)
    X_train = split['X_train']
    X_test  = split['X_test']

    # Check 1: fit_scaler returns StandardScaler
    try:
        assert 'fit_scaler' in globals()
        scaler = fit_scaler(X_train)
        assert isinstance(scaler, StandardScaler), \
            f'expected StandardScaler, got {type(scaler).__name__}'
        assert hasattr(scaler, 'mean_'), 'scaler must be fitted (has mean_ attribute)'
        passed += 1; print('\u2705 Check 1: fit_scaler returns fitted StandardScaler')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: scale_features returns DataFrame with same columns
    try:
        assert 'scale_features' in globals()
        X_scaled = scale_features(scaler, X_train)
        assert isinstance(X_scaled, pd.DataFrame), \
            f'expected DataFrame, got {type(X_scaled).__name__}'
        assert list(X_scaled.columns) == list(X_train.columns), \
            'column names must be preserved after scaling'
        passed += 1; print(f'\u2705 Check 2: scale_features returns DataFrame with {len(X_scaled.columns)} cols')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: training features have mean ≈ 0 after scaling
    try:
        col_means = X_scaled.mean()
        max_mean  = col_means.abs().max()
        assert max_mean < 1e-9, \
            f'scaled train means should be ≈0; max abs mean = {max_mean:.2e}'
        passed += 1; print(f'\u2705 Check 3: scaled training means ≈ 0 (max abs = {max_mean:.2e})')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: training features have std ≈ 1 after scaling
    try:
        col_stds = X_scaled.std(ddof=0)  # population std matches scaler
        max_err  = (col_stds - 1).abs().max()
        assert max_err < 1e-9, \
            f'scaled train stds should be ≈1; max abs error = {max_err:.2e}'
        passed += 1; print(f'\u2705 Check 4: scaled training stds ≈ 1 (max err = {max_err:.2e})')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: test set can be transformed with the same scaler
    try:
        X_test_scaled = scale_features(scaler, X_test)
        assert list(X_test_scaled.columns) == list(X_train.columns), \
            'test scaled columns must match train columns'
        assert X_test_scaled.shape == X_test.shape, \
            f'shape mismatch: {X_test_scaled.shape} vs {X_test.shape}'
        passed += 1; print(f'\u2705 Check 5: test set transformed, shape={X_test_scaled.shape}')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
from sklearn.preprocessing import StandardScaler


def fit_scaler(X_train: pd.DataFrame) -> StandardScaler:
    """Fit a StandardScaler on training data only."""
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler


def scale_features(scaler: StandardScaler,
                   X: pd.DataFrame) -> pd.DataFrame:
    """Transform X using a fitted scaler; return DataFrame with same columns."""
    scaled = scaler.transform(X)
    return pd.DataFrame(scaled, columns=X.columns, index=X.index)
```

</details>